# 🍕🚀 Antonia's Pizza — AI Video Generator (Colab)

**"The Future of Delivery is Here."**

This notebook renders the flying pizza-box reveal. It picks the model that fits your GPU automatically:

| Your GPU | What runs | Output |
|---|---|---|
| **T4 (free Colab, 16 GB)** | Wan 2.1 · T2V-1.3B | 832×480 · 5 s |
| **A100 / L4 / 24 GB+** | Wan 2.2 · TI2V-5B | 1280×704 · 5 s, **uses `keyframe_hero.png` as frame 0** |

### How to use
1. `Runtime ▸ Change runtime type ▸ T4 GPU` (or A100 if you have Pro).
2. `Runtime ▸ Run all`. First run downloads ~15–35 GB of weights — be patient.
3. The MP4 lands in `/content/out/` and downloads automatically.

> ⚠️ **Reality check:** no open model renders 23 cinematic seconds in one pass. Each run = one 5-second shot.
> Generate Shot 01–04 (see `shotlist.md`), then cut them together and add the end card in CapCut.
> The end card text is **not** AI — real typography always looks better than generated letters.

In [ ]:
# @title 1) Check GPU + install dependencies { display-mode: "form" }
import os, subprocess, sys, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Go to Runtime ▸ Change runtime type ▸ T4 GPU, then Run all again.")

gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB")
print(f"   bf16 supported: {torch.cuda.is_bf16_supported()}")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip -q install -U "diffusers>=0.35" "accelerate>=1.1" "transformers>=4.49" imageio imageio-ffmpeg ftfy
print("\n✅ dependencies installed")

In [ ]:
# @title 2) Settings — the only cell you need to edit { display-mode: "form" }
import os, torch

# "auto"    -> picks quality when VRAM >= 22 GB, otherwise budget
# "quality" -> Wan 2.2 TI2V-5B  (needs ~16-24 GB, works with CPU offload)
# "budget"  -> Wan 2.1 T2V-1.3B (runs on a free T4)
MODE = "auto"

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
if MODE == "auto":
    MODE = "quality" if VRAM_GB >= 22 else "budget"
print("MODE =", MODE)

# ---- the shot you are rendering (Shot 01 by default) -------------------------
PROMPT = (
    "Low-angle hero shot, slow motion. A giant Antonia's Pizza box sits on a glossy black "
    "stage floor, lid slightly open, golden steam pouring out. It lifts off the ground and rises "
    "two feet, tilting as it hovers. Crimson and amber spotlights sweep across it, embers and dust "
    "drifting through the beams, ground fog rolling. Stunned silhouetted crowd at the edges. Camera "
    "pushes in low, slight handheld. Photoreal, anamorphic lens flare, shallow depth of field, "
    "24fps, film grain."
)

NEGATIVE = (
    "blurry, low quality, distorted text, misspelled lettering, warped logo, extra boxes, "
    "melting cardboard, watermark, jitter, static shot, cartoon, plastic look, oversaturated"
)

SEED   = 42
OUTDIR = "/content/out"
os.makedirs(OUTDIR, exist_ok=True)

# ---- keyframe (optional, quality mode only) ---------------------------------
KEYFRAME = "/content/keyframe_hero.png"
uploaded  = False
if not os.path.exists(KEYFRAME):
    try:
        from google.colab import files as colab_files
        print("⬆️  Upload `keyframe_hero.png` (from the deliverables folder) for image-to-video,")
        print("    or press Cancel to run text-to-video only.")
        up = colab_files.upload()
        if up:
            for name in up:
                os.rename(name, KEYFRAME)
            uploaded = True
    except Exception as e:
        print("no upload:", e)

print("keyframe available:", os.path.exists(KEYFRAME))

In [ ]:
# @title 3) QUALITY — Wan 2.2 TI2V-5B (720p, image-to-video) { display-mode: "form" }
# Needs ~16 GB with CPU offload, ~24 GB comfortable. A100 / L4 / 4090 territory.
# Skips itself automatically if you are in budget mode.
import os, time, torch

if MODE != "quality":
    print("⏭️  skipped (budget mode)")
else:
    from diffusers import WanImageToVideoPipeline, WanPipeline
    from diffusers.utils import export_to_video, load_image

    MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
    print("loading", MODEL)
    t0 = time.time()

    pipe = WanImageToVideoPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()          # keeps T4/L4 alive; remove on 80 GB
    pipe.vae.enable_tiling()

    H, W, FRAMES = 704, 1280, 121            # 121 frames @ 24 fps ≈ 5 s (5B model's native size)

    kwargs = dict(
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        height=H, width=W, num_frames=FRAMES,
        guidance_scale=5.0,
        num_inference_steps=50,
        generator=torch.Generator(device="cpu").manual_seed(SEED),
    )

    if os.path.exists(KEYFRAME):
        print("🖼️  image-to-video using keyframe_hero.png")
        image = load_image(KEYFRAME).resize((W, H))
        video = pipe(image=image, **kwargs).frames[0]
    else:
        print("📝 text-to-video (no keyframe found)")
        t2v = WanPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
        t2v.enable_model_cpu_offload(); t2v.vae.enable_tiling()
        video = t2v(**kwargs).frames[0]

    out = f"{OUTDIR}/shot01_wan22_5b.mp4"
    export_to_video(video, out, fps=24)
    print(f"✅ {out}  ({time.time()-t0:.0f}s total)")

    del pipe
    torch.cuda.empty_cache()

In [ ]:
# @title 4) BUDGET — Wan 2.1 T2V-1.3B (free Colab T4) { display-mode: "form" }
# 832x480, 81 frames ≈ 3.4 s. Text-to-video only: the 1.3B model has no image conditioning.
# Brand accuracy on the box is handled in post-production (real logo overlay).
import time, torch

if MODE != "budget":
    print("⏭️  skipped (quality mode)")
else:
    from diffusers import WanPipeline
    from diffusers.utils import export_to_video

    MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
    print("loading", MODEL)
    t0 = time.time()

    pipe = WanPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()

    video = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        height=480, width=832, num_frames=81,
        guidance_scale=5.0,
        num_inference_steps=40,
        generator=torch.Generator(device="cpu").manual_seed(SEED),
    ).frames[0]

    out = f"{OUTDIR}/shot01_wan21_1.3b.mp4"
    export_to_video(video, out, fps=16)
    print(f"✅ {out}  ({time.time()-t0:.0f}s total)")

    del pipe
    torch.cuda.empty_cache()

In [ ]:
# @title 5) Preview + download { display-mode: "form" }
import glob, os
from IPython.display import Video, display

clips = sorted(glob.glob(f"{OUTDIR}/*.mp4"))
print("videos:", clips)

for c in clips:
    print(c, f"{os.path.getsize(c)/1e6:.1f} MB")
    display(Video(c, embed=True, width=720))

try:
    from google.colab import files as colab_files
    for c in clips:
        colab_files.download(c)
except Exception:
    pass

In [ ]:
# @title 🔁 Tips & troubleshooting { display-mode: "form" }
print('''
OUT OF MEMORY
  - quality mode: lower W/H to 480x832 and FRAMES to 81
  - budget mode:  lower to 416x720 and FRAMES to 49
  - restart runtime before switching modes, VRAM does not always free itself

TEXT LOOKS WRONG ON THE BOX
  - that is normal: open video models cannot spell reliably.
  - fix it in post: overlay the real logo (post_production.md), or use
    image-to-video with a clean keyframe you designed in Canva/Photoshop.

WANT MORE THAN 5 SECONDS
  - render Shot 01 -> 04 separately and cut them (shotlist.md)
  - or chain: last frame of clip N becomes the keyframe of clip N+1
''')